# Reinforcement Learning

## RL Overview
![RL Diagram](https://inst.eecs.berkeley.edu/~cs188/textbook/assets/images/feedback-loop.png)
- Main Idea:
  - Recieve feedback in the form of rewards
  - Agents utility is defined by the reward function.
    - Utility in this context is the total accumulated value over an entire (possibly infinite) future trajectory
  - Must learn to act so as to maximize expected rewards
  - All learning is based on observed samples of outcomes.
- We still assume an MDP:
  - Set of States s ϵ S
  - Set of actions (per state) A
  - Model T(s,a,s')
    - aka transition model, dynamics
  - Reward Function R(s,a,s')
- Still looking to define a policy π(s)
- The **new twist** is that we don't know T or R
  - i.e. we don't know which states are good (their rewards) or what the actions do (their transitions)
  - We **must** actually try out actions and states to learn.
- **Offline** - What MDPs are; they think about their next action and do not have to learn by trying beause they already know the result of the action.
- **Online** - What RL does; basically learning by action because it has no prior knowledge of the rewards associate with the actions.



## Model-Based Learning
- Learn and curate an approximate model based on experiences.
- Solve for values as if the learned model were correct.

- **Step 1**: Learn Empirical MDP Model (defining the approximate T's and R's)
  - count outcomes s' for each (s,a)
  - Normalize to give estimate of $\hat{T}(s,a,s')$
  - Discover each $\hat{R}(s,a,s')$ when we experience (s,a,s')
- **Step 2**: Solve the learned MDP
  - for example, use value iteration as before
- Note: x represents exit (terminal) state
  - γ - gamma

![s](https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEjIWiVYovr0aRgZzuzBSfai9qGr0b-ztdAG7rVGQ7UTzLZlRpPMtDJaSjl2eAMnmU5WY1ZbkZf3IXmEyJbAnNgCvUPHP-jSsKz5en-GNYiiHCZ9cMK0qen2Rk4lpsXPdQXC9-Hv_anDR4SY/w543-h241/image.png)  
- P(a) is the probability distribution
- Model free version works for this example because the average of the samples naturally converges to the true expected value.

## Passive RL
- Thought is to watch a recording of someone else failing and learn from that.
- Simplifyfing the task by making overall RL problem much easier: policy evaluation
  - Input: fixed policy π(s)
  - You don't know the transitions or rewards
  - Goal is to learn the state values by **doing and following the fixed policy**
- In this situation:
  - Learner is "along for the ride"
  - No choice about what actions to take, simply executing the policy and learning from experience
  - **NOT OFFLINE PLANNING** - You actually take actions in the world

### Direct Evaluation
- Model free, passive RL approach
- Goal: compute values for each state under π (policy)
- Idea: average together observed sample values
  - Act according to π (policy)
  - Every time a state visited, write down what the sum of *discounted* rewards turned out to be
  - Average those samples
- Pros:
  - Easy to understand
  - doesn't require T or R knowledge
  - Eventually computes the correct average values using just sample transitions.
- Cons:
  - Wastes info about state connects
  - each state much be learned seperately -> takes a long time to learn

### Policy Evaluation
- Recall simplified Bellman updates calculate V for a fixed policy:
  - Each round, replace V with a one-step-look-ahead layer over V
  - $V_0^\pi(s) = 0$
  - $V_{k+1}^{\pi_i}(s) \leftarrow \sum_{s'} T(s, \pi_i(s), s') \left[ R(s, \pi_i(s), s') + \gamma V_k^{\pi_i}(s') \right]$
  - This approach fully exploited the connections between the states.
  - Unfortunately, we need T and R to do it!
- So, **how can we do this update to V without knowing T and R?**
  - how to take a weighted average without knowing the weights?

#### Sample-Based Policy Evaluation?
- We want to improve our estimate of V by coputing these averages:
  - $V_{k+1}^{\pi_i}(s) \leftarrow \sum_{s'} T(s, \pi_i(s), s') \left[ R(s, \pi_i(s), s') + \gamma V_k^{\pi_i}(s') \right]$

- So we take samples of outcomes s' (by doing the action) and average.
$$
\begin{align*}
\text{sample}_1 &= R(s, \pi(s), s'_1) + \gamma V_k^\pi(s'_1) \\
\text{sample}_2 &= R(s, \pi(s), s'_2) + \gamma V_k^\pi(s'_2) \\
&\vdots \\
\text{sample}_n &= R(s, \pi(s), s'_n) + \gamma V_k^\pi(s'_n)
\end{align*}
$$
$$
V_{k+1}^\pi(s) \leftarrow \frac{1}{n} \sum_{i=1}^{n} \text{sample}_i
$$

- This works and is needed because in real MDPs, when you take an action `a = π(s)` from state s, the environment does not always send you to th esame next state. It is ususally:
  - With some probability you go to one state s'
  - With another probability you go to a different state s''

### Temporal Difference Learning
- Model free, passive RL approach
- Main Idea: **learn from every experience**
  - update V(s) everytime we experience a transition (s,a,s',r)
  - likely outcomes s' will contribute updates more often.
  - Are updating the value of the current state (V(s)) based on the value of the next state (V(s')).
- How temporal difference learns values:
  - policy is still fixed, still doing evaluation of actions
  - Moves values towards the value of whatever successor occurs on that action (running average)
  - Sample of V(s) :$sample = R(s, \pi(s), s') + \gamma V^\pi(s')$
  - Update to V(s): $V^\pi(s) \leftarrow (1 - \alpha) V^\pi(s) + \alpha \cdot sample$
    - Same update can also be written as (mathematically equivilent): $V^\pi(s) \leftarrow V^\pi(s) + \alpha \big( sample - V^\pi(s) \big)$
    - α is the learning rate - controls how much we trust the new sample vs our old estimate

- The update rule above is called "Exponential Moving Average"
  - It makes the recent samples more important.
  - Forgets about the past (distant past values were wrong anyway)
  - Decreasing the learning rate (α) can give converging averages.
- Recall: gamma (γ) determines how much the agent cares about future rewards compared to immediate ones

#### Problems with TDVL
- Recall that it is a model-free way to do policy evaluation, mimicking Bellman updates with running sample averages.
- We can't turn values into a new policy because this teq gives you no information about which actions are 'good' or 'bad'


## Active RL
- Aka full reinforcement learning: optimal policies
  - You don't know Ts or Rs
  - You choose the actions now.
  - Goal: learn the optimal policy/values

- Learner makes choices
- Fundamental tradeoff between exploration and exploitation
- ONLINE PLANNING; actions are taken

## Q-Value iteration
- Recall:
  - Value iteration: find successive (depth-limited) values.
  - Start with $V_0(s) = 0$ which we know is right.
  - Given $V_k$, calculate the depth k+1 values for all states
  - $V_{k+1}(s) <- \max_a \sum_{s'} T(s, a, s') \left[ R(s, a, s') + \gamma V_k(s') \right]$

- But, Q-values are more useful, so compute them instead
  - Recall: Q-value answers the quesiton: "If I am in state s and I choose to take this specific action a, how good will my future be (on average)
  - *q state* - a state and action pair (s,a)

- Start with $Q_0(s,a) = 0$, which we know is right
- Given $Q_k$, calculate the depth k+1 q-values for all q-states
  - $Q_{k+1}(s, a) \leftarrow \sum_{s'} T(s, a, s') \Big[ R(s, a, s') + \gamma \max_{a'} Q_k(s', a') \Big]$
  - Note: $\max_{a'} Q_k = V_k(s')$

### Q-Learning
- Sample based Q-value iteraiton
  - $Q_{k+1}(s, a) \leftarrow \sum_{s'} T(s, a, s') \Big[ R(s, a, s') + \gamma \max_{a'} Q_k(s', a') \Big]$
  - Note: $\max_{a'} Q_k = V_k(s')$
- Learn Q(s,a) values as you go.
  - Receive a sample (s,a,s',r)
  - Consider your old estimate: Q(s,a)
  - Consider your new sample estimate: $sample = [ R(s, a, s') + \gamma \max_{a'} Q_k(s', a')]$
  - Incorporate the new estimate into a running average
    - $Q(s,a) ← (1-α)Q(s,a) + (α) [sample]$
- Returns an amazing result as Q-learning converges to optimal policy, even if your acting suboptimally.
  - 'even while its behaving according to a different (possibly suboptimal or random) policy.
  - This is called **off-policy learning**
- There are some Caveats:
  - You must explore enough
  - Must eventually make the learning rate small enough while not decreasing it too quickly
  - In the limit, it doesn't matter how you select actions
    - in the limit just means 'as you go on forever'

### How to Explore in Q Learning?

- Simplest: random actions (ϵ-greedy)
  - Every time step, flip a coin
  - With (small) probability ϵ,act randomly
  - With (large) probability 1-ϵ, act on current policy
  - Common problem is you eventually explore the entire space, but keep thrashing around once learning is done.
    - Can solve by lowering ϵ over time.
    - Can also solve via *exploration funcitons*
### Exploration Functions
- Random actions: explore a fixed amount
- Better idea: explore areas whose badness (its a bad place to be) is not (yet) established, eventually stop exploring
- Takes value estimate u and a visit count n, returning an optimistic utility
  - optimistic - better than it actually is.
  - $f(u,n) = u + k/n$
    - 'value' - q value estimate
    - utility is an agents value over an entire path
  - Regular Q-Update: $Q(s, a) \leftarrow \alpha \Big[ R(s, a, s') + \gamma \max_{a'} Q(s', a') \Big]$
  - Modified Q-Update:
    - $Q(s, a) \leftarrow \alpha \Big[ R(s, a, s') + \gamma \max_{a'} f\big(Q(s', a'), N(s', a')\big) \Big]$
      - N - visit count.
      - **a'** - every possible action the agent could take in the next state s'
  - Note that in this exploration function, a rarly visited state has a high, f(Q(s', a'), N(s', a')).
    - Therefore, any previohus state that can lead to this under-explored (s',a') will also see a higher Q-Value.

### Regret
- Even when learning the optimal policy, mistakes are still made.
- *Regret* - measure of total mistake cost.
  - aka the difference between your (expected) rewards, including youthful suboptimallity and optimal expected rewards
- Minimizing regret goes beyond learning to be optimal - it requires *optimally learning to be optimal*.
- Random exploration and exploration functions both end up optimal.
  - **But random exploration has higher regret**


## Approximate Q Learning
- Basic Q-Learning keeps a table of all q-values.
  - But realilstically, we cannot possiblyy learn about every single state
- So instead, we want to generalize (approximate) - basically ML
  - 1. Learn about some small number of training states from experience.
  - Generalize that experience to new, similar situations.

![s](https://i.imgur.com/33PdXoj.png)
- Note the one pellet difference on the right.
- We can solve this issue with Feature-Based Representation  
  - Basically just hand selecting the features that matter (like in ML )
  You can descripe a q-state with features as well.

### Linear Value Functions
- In this feature framework, we can write a q (or any value function) using a few weights:

- Goal: learn the weights w
  - advantage: experience summed up in a few powerful numbers
  - $V(s) = w_1 f_1(s) + w_2 f_2(s) + \dots + w_n f_n(s)$
  - $Q(s, a) = w_1 f_1(s,a) + w_2 f_2(s,a) + \dots + w_n f_n(s,a)$
  - Disadvantage: states may share features but actually be very different in true value.
    - The gamble you take by hand selecting features

### Q-Learning with Linear Q-funcitons
- this is online least squares
- $Q(s, a) = w_1 f_1(s,a) + w_2 f_2(s,a) + \dots + w_n f_n(s,a)$
- transition = (s,a,r,s')
  - Tuple of things that happen together in one time step.

- Order of operations:
  - Calculate current Q(s,a) using current weights
  - Compute the difference
    - $\text{difference} = \left[ r + \gamma \max_{a'} Q(s', a') \right] - Q(s, a)$
  - Update both the Q value and each induvidual weight (that correlates to a feature) as such:
    - $Q(s, a) \leftarrow Q(s, a) + \alpha [\text{difference}]$
    - $w_i \leftarrow w_i + \alpha [\text{difference}] f_i(s, a)$    

- Least Squares (total) error:
  - $\text{total error} = \sum_i (y_i - \hat{y}_i)^2 = \sum_i \left( y_i - \sum_k w_k f_k(x_i) \right)^2$
- Can minimize the error using gradient descent. Notation for the Q learning context as follows:

  - $\text{error}(w) = \frac{1}{2} \left( y - \sum_k w_k f_k(x) \right)^2$

  - $\frac{\partial \text{error}(w)}{\partial w_m} = - \left( y - \sum_k w_k f_k(x) \right) f_m(x)$

  - $w_m \leftarrow w_m + \alpha \left( y - \sum_k w_k f_k(x) \right) f_m(x)$

- Approximate q update explained:

  - $w_m \leftarrow w_m + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right] f_m(s, a)$
  - target - $r + \gamma \max_{a'} Q(s', a')$
  - prediction - $Q(s, a)$

- Limiting the capacity (amount of features analyzed) can help reduce overfitting.